# Decision Tree Analysis - Lab 02 - EIF420O Artificial Intelligence
## Universidad Nacional de Costa Rica
### Dataset: BankChurners (Bank Customer Churn Prediction)
**Target:** Attrition_Flag (Existing Customer / Attrited Customer)

This notebook implements the Decision Tree classification analysis using the Supervisado class pattern.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

## 1. Supervisado Class (Course Package)

In [ ]:
class Supervisado:
    def __init__(self, df):
        self.__df = df

    @property
    def df(self):
        return self.__df

    @df.setter
    def df(self, p_df):
        self.__df = p_df

    def preparar_datos(self, target_col='target', test_size=0.25, random_state=42):
        # No scaling for DT (tree-based, scale-invariant)
        X = self.__df.drop(columns=[target_col])
        y = self.__df[target_col]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y)
        return X_train, X_test, y_train, y_test, y

    def modeloDT(self, X_train, y_train, criterion='gini', max_depth=None,
                 min_samples_split=2, min_samples_leaf=1, random_state=42):
        model = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth,
                                       min_samples_split=min_samples_split,
                                       min_samples_leaf=min_samples_leaf,
                                       random_state=random_state)
        model.fit(X_train, y_train)
        return model

    def predecir(self, model, X_test):
        return model.predict(X_test)

    def evaluar(self, y_test, y_pred, y):
        MC = confusion_matrix(y_test, y_pred)
        return self.indices_general(MC, list(np.unique(y)))

    def indices_general(self, MC, nombres=None):
        precision_global = np.sum(MC.diagonal()) / np.sum(MC)
        error_global = 1 - precision_global
        precision_categoria = pd.DataFrame(MC.diagonal() / np.sum(MC, axis=1)).T
        if nombres is not None:
            precision_categoria.columns = nombres
        return {"Matriz de Confusión": MC, "Precisión Global": precision_global,
                "Error Global": error_global, "Precisión por categoría": precision_categoria}

## 2. Data Loading and Preparation

In [ ]:
df = pd.read_csv('BankChurners.csv')
df = df.drop(columns=['ID'])
le_target = LabelEncoder()
df['target'] = le_target.fit_transform(df['Attrition_Flag'])
df = df.drop(columns=['Attrition_Flag'])
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

print("Target classes:", le_target.classes_)
print("Distribution:", df['target'].value_counts().to_dict())
print("Shape:", df.shape)
df.head()

## 3. Standard Decision Tree (gini, unrestricted)

In [ ]:
sup = Supervisado(df)
X_train, X_test, y_train, y_test, y = sup.preparar_datos(target_col='target', random_state=42)
feature_names = list(X_train.columns)

model_std = sup.modeloDT(X_train, y_train, criterion='gini', max_depth=None,
                         min_samples_split=2, min_samples_leaf=1)
y_pred_std = sup.predecir(model_std, X_test)
idx_std = sup.evaluar(y_test, y_pred_std, y)

print(f"Precision: {idx_std['Precisión Global']:.4f}")
print(f"Error: {idx_std['Error Global']:.4f}")
print(f"By category:\n{idx_std['Precisión por categoría']}")
print(f"Depth: {model_std.get_depth()}, Leaves: {model_std.get_n_leaves()}")
print(f"\nCM:\n{idx_std['Matriz de Confusión']}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(confusion_matrix=idx_std['Matriz de Confusión'],
                       display_labels=le_target.classes_).plot(ax=ax, cmap='Blues')
ax.set_title('CM - Standard DT (gini, unrestricted)')
plt.tight_layout(); plt.show()

## 4. Systematic Hyperparameter Exploration

In [ ]:
resultados = []
for crit in ['gini', 'entropy']:
    for d in range(1, 21):
        for mss in [2, 5, 10, 15, 20]:
            for msl in [1, 2, 5, 10]:
                model = sup.modeloDT(X_train, y_train, criterion=crit,
                                     max_depth=d, min_samples_split=mss, min_samples_leaf=msl)
                y_pred = sup.predecir(model, X_test)
                idx = sup.evaluar(y_test, y_pred, y)
                resultados.append({'criterion': crit, 'max_depth': d,
                                   'min_samples_split': mss, 'min_samples_leaf': msl,
                                   'precision_global': idx['Precisión Global'],
                                   'error_global': idx['Error Global'],
                                   'y_pred': y_pred, 'model': model})

df_results = pd.DataFrame([{k:v for k,v in r.items() if k not in ['y_pred','model']} for r in resultados])
print(f"Configs: {len(df_results)}, Best: {df_results['precision_global'].max():.4f}, Worst: {df_results['precision_global'].min():.4f}")

## 5. Visualizations

In [ ]:
# Precision vs depth by criterion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, crit in enumerate(['gini', 'entropy']):
    ax = axes[i]
    sub = df_results[(df_results['criterion']==crit) & (df_results['min_samples_leaf']==1)]
    for mss in [2, 5, 10]:
        d = sub[sub['min_samples_split']==mss]
        ax.plot(d['max_depth'], d['precision_global'], marker='o', markersize=4, label=f'min_split={mss}')
    ax.set_xlabel('max_depth'); ax.set_ylabel('Precision'); ax.set_title(f'Criterion: {crit}')
    ax.legend(); ax.grid(alpha=0.3); ax.set_xticks(range(1,21))
fig.suptitle('DT: Precision vs max_depth', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Heatmap
best_crit = df_results.groupby('criterion')['precision_global'].mean().idxmax()
sub = df_results[(df_results['criterion']==best_crit) & (df_results['min_samples_leaf']==1)]
pivot = sub.pivot_table(index='min_samples_split', columns='max_depth', values='precision_global')
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title(f'DT Heatmap (criterion={best_crit})', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Top 10
top10 = df_results.nlargest(10, 'precision_global').reset_index(drop=True)
top10['label'] = top10.apply(lambda r: f"d={r['max_depth']}, {r['criterion']}, split={r['min_samples_split']}, leaf={r['min_samples_leaf']}", axis=1)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(range(len(top10)), top10['precision_global'],
               color=sns.color_palette('magma', len(top10)), edgecolor='black')
ax.set_yticks(range(len(top10))); ax.set_yticklabels(top10['label'])
ax.set_xlabel('Global Precision'); ax.set_title('Top 10 DT Configurations', fontweight='bold')
for bar, val in zip(bars, top10['precision_global']):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center')
ax.invert_yaxis(); plt.tight_layout(); plt.show()
top10[['criterion','max_depth','min_samples_split','min_samples_leaf','precision_global']]

## 6. Best Model Analysis

In [ ]:
idx_best = df_results['precision_global'].idxmax()
best = resultados[idx_best]
best_indices = sup.evaluar(y_test, best['y_pred'], y)
print(f"Best: criterion={best['criterion']}, depth={best['max_depth']}, split={best['min_samples_split']}, leaf={best['min_samples_leaf']}")
print(f"Precision: {best_indices['Precisión Global']:.4f}")
print(f"Error: {best_indices['Error Global']:.4f}")
print(f"By category:\n{best_indices['Precisión por categoría']}")
print(f"\nCM:\n{best_indices['Matriz de Confusión']}")
print(f"\n{classification_report(y_test, best['y_pred'], target_names=le_target.classes_)}")

In [ ]:
# Side-by-side CMs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(confusion_matrix=idx_std['Matriz de Confusión'],
                       display_labels=le_target.classes_).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Standard DT')
ConfusionMatrixDisplay(confusion_matrix=best_indices['Matriz de Confusión'],
                       display_labels=le_target.classes_).plot(ax=axes[1], cmap='Greens')
axes[1].set_title(f"Best DT (depth={best['max_depth']}, {best['criterion']})")
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
imp = best['model'].feature_importances_
idx_sort = np.argsort(imp)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(idx_sort)), imp[idx_sort], color=sns.color_palette('crest', len(idx_sort)), edgecolor='black')
ax.set_yticks(range(len(idx_sort))); ax.set_yticklabels(np.array(feature_names)[idx_sort])
ax.set_xlabel('Importance'); ax.set_title('Feature Importance - Best DT', fontweight='bold')
ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Tree structure (top 4 levels)
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(best['model'], feature_names=feature_names, class_names=list(le_target.classes_),
          filled=True, rounded=True, ax=ax, fontsize=7, proportion=True, max_depth=4)
ax.set_title('Decision Tree Structure (top 4 levels)', fontweight='bold')
plt.tight_layout(); plt.show()